# Wrapping a Model in FastAPI

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/02-wrapping-model-in-fastapi.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.

> **Note:** This lesson builds a service that must run as a long-lived process. The cells below show the full implementation but cannot run end-to-end in Colab. To run this lesson: clone the repo locally and use `uvicorn main:app` or `docker compose up`.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You have a working model call. Now a frontend team wants to use it. A mobile app wants to use it. A partner integration wants to use it. They all speak HTTP.

The naive approach is to run a script and expose it with something like `flask run` or a simple `http.server`. That breaks under any real load, gives you no request validation, no structured error responses, and no way to tell if the service is alive.

FastAPI solves each of these. It gives you: automatic request validation (Pydantic), automatic OpenAPI docs, async request handling, clean error responses, and a standard pattern for resource initialization. It is the right tool for this job and it is genuinely fast.

The one mistake eng...

## The Concept

### Request Flow Through FastAPI



### Lifespan: Where the Client Lives

```
Process starts
    |
    v
lifespan() -- startup phase
    |-- anthropic.Anthropic() created ONCE
    |-- stored in app.state
    |-- connection pool initialized
    v
Service accepts requests
    |-- route handlers read from app.state
    |-- no new client per request
    v
lifespan() -- shutdown phase
    |-- clean up resources
    v
Process ends
```

### HTTP Status Codes for AI Services

```
+--------+---------------------------+---------------------------------------+
| Code   | Name                      | When ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 06-02: Wrapping a Model in FastAPI

A production-ready FastAPI service wrapping the Anthropic API with:
- Lifespan event for single client initialization
- Pydantic request/response models with automatic validation
- Two model endpoints: /generate and /extract
- Health check endpoint
- Correct HTTP status codes and structured error responses

Usage:
    pip install fastapi uvicorn anthropic pydantic
    export ANTHROPIC_API_KEY=sk-ant-...
    uvicorn main:app --reload --port 8000

Test:
    curl http://localhost:8000/health
    curl -X POST http://localhost:8000/generate \
        -H "Content-Type: application/json" \
        -d '{"prompt": "What is the capital of France?"}'
"""

import json
import logging
import os
from contextlib import asynccontextmanager
from datetime import datetime, timezone

import anthropic
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)
log = logging.getLogger(__name__)

### Request and response models

In [ ]:
class GenerateRequest(BaseModel):
    prompt: str = Field(
        ...,
        min_length=1,
        max_length=4000,
        description="The user prompt to send to the model",
    )
    max_tokens: int = Field(
        default=512,
        ge=1,
        le=4096,
        description="Maximum tokens in the response",
    )
    system: str | None = Field(
        default=None,
        max_length=2000,
        description="Optional system prompt to guide the model's behavior",
    )


class GenerateResponse(BaseModel):
    text: str
    input_tokens: int
    output_tokens: int
    model: str


class ExtractRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=8000,
        description="The text to extract structured data from",
    )
    schema_hint: str = Field(
        ...,
        min_length=1,
        max_length=500,
        description="Description of what to extract, e.g. 'name, email, company'",
    )


class ExtractResponse(BaseModel):
    raw_json: str
    parsed: dict | None  # None if the model output was not valid JSON
    input_tokens: int
    output_tokens: int


class HealthResponse(BaseModel):
    status: str
    model: str
    timestamp: str

### Lifespan: initialize the Anthropic client ONCE at startup

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    """
    FastAPI lifespan event handler.

    Everything before 'yield' runs at startup.
    Everything after 'yield' runs at shutdown.

    The Anthropic client is created here -- once per process -- and stored in
    app.state so route handlers can access it without creating new instances.
    """
    # STARTUP
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "ANTHROPIC_API_KEY is not set. "
            "Export it before starting the service: export ANTHROPIC_API_KEY=sk-ant-..."
        )

    model = os.environ.get("MODEL", "claude-3-5-haiku-20241022")
    timeout = float(os.environ.get("TIMEOUT_SECONDS", "30.0"))

    app.state.client = anthropic.Anthropic(
        api_key=api_key,
        timeout=timeout,
        max_retries=2,
    )
    app.state.model = model

    log.info("Startup complete: model=%s timeout=%.1fs", model, timeout)

    yield  # the service is alive; requests are handled here

    # SHUTDOWN
    log.info("Shutdown: cleaning up resources")

### Application

In [ ]:
app = FastAPI(
    title="AI Service",
    description="Production FastAPI wrapper around the Anthropic API",
    version="1.0.0",
    lifespan=lifespan,
)

EXTRACT_SYSTEM = (
    "You are a data extraction assistant. "
    "Extract the requested fields from the provided text and return them as a JSON object. "
    "Return ONLY the JSON object with no preamble, explanation, or markdown code fences."
)

### Routes

In [ ]:
@app.get("/health", response_model=HealthResponse, tags=["ops"])
async def health(req: Request) -> HealthResponse:
    """
    Health check for load balancers and uptime monitors.

    Returns 200 when the service is running and configured.
    Does NOT call the model -- health checks must be fast and cheap.
    Load balancers call this every 10-30 seconds.
    """
    return HealthResponse(
        status="ok",
        model=req.app.state.model,
        timestamp=datetime.now(timezone.utc).isoformat(),
    )


@app.post("/generate", response_model=GenerateResponse, tags=["model"])
async def generate(req: Request, body: GenerateRequest) -> GenerateResponse:
    """
    Generate a text response from the model.

    Returns 200 on success.
    Returns 422 automatically if the request body fails Pydantic validation.
    Returns 429 if the upstream API is rate-limited.
    Returns 500 on unexpected errors.
    Returns 502 on upstream API errors.
    """
    client: anthropic.Anthropic = req.app.state.client
    model: str = req.app.state.model

    log.info(
        "POST /generate model=%s prompt_chars=%d max_tokens=%d",
        model,
        len(body.prompt),
        body.max_tokens,
    )

    kwargs: dict = {
        "model": model,
        "max_tokens": body.max_tokens,
        "messages": [{"role": "user", "content": body.prompt}],
    }
    if body.system:
        kwargs["system"] = body.system

    try:
        response = client.messages.create(**kwargs)
    except anthropic.APIStatusError as e:
        log.error("Anthropic API status error: status=%d message=%s", e.status_code, e)
        if e.status_code == 429:
            raise HTTPException(
                status_code=429,
                detail="Rate limit reached. Wait a moment and retry.",
            )
        raise HTTPException(
            status_code=502,
            detail=f"Upstream model error (HTTP {e.status_code}).",
        )
    except anthropic.APITimeoutError as e:
        log.error("Anthropic API timeout: %s", e)
        raise HTTPException(
            status_code=504,
            detail="Model request timed out. Retry with a shorter prompt or higher timeout.",
        )
    except Exception as e:
        log.error("Unexpected error in /generate: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Internal server error.")

    result = GenerateResponse(
        text=response.content[0].text,
        input_tokens=response.usage.input_tokens,
        output_tokens=response.usage.output_tokens,
        model=response.model,
    )
    log.info(
        "POST /generate success input_tokens=%d output_tokens=%d",
        result.input_tokens,
        result.output_tokens,
    )
    return result


@app.post("/extract", response_model=ExtractResponse, tags=["model"])
async def extract(req: Request, body: ExtractRequest) -> ExtractResponse:
    """
    Extract structured data from text using the model.

    Returns parsed JSON when the model output is valid JSON.
    Returns raw_json with parsed=null when the model output is not valid JSON.
    Stripes markdown code fences from model output before attempting to parse.
    """
    client: anthropic.Anthropic = req.app.state.client
    model: str = req.app.state.model

    log.info(
        "POST /extract model=%s text_chars=%d schema=%r",
        model,
        len(body.text),
        body.schema_hint,
    )

    user_message = (
        f"Text to extract from:\n{body.text}\n\n"
        f"Extract these fields: {body.schema_hint}\n\n"
        f"Return a JSON object with those fields and nothing else."
    )

    try:
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=EXTRACT_SYSTEM,
            messages=[{"role": "user", "content": user_message}],
        )
    except anthropic.APIStatusError as e:
        log.error("Anthropic API status error in /extract: status=%d", e.status_code)
        raise HTTPException(
            status_code=502,
            detail=f"Upstream model error (HTTP {e.status_code}).",
        )
    except Exception as e:
        log.error("Unexpected error in /extract: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Internal server error.")

    raw = response.content[0].text.strip()

    # Strip markdown code fences if the model wrapped the JSON
    # e.g. ```json\n{...}\n``` or ```\n{...}\n```
    if raw.startswith("```"):
        lines = raw.split("\n")
        if len(lines) > 2:
            raw = "\n".join(lines[1:-1]).strip()

    parsed: dict | None = None
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        log.warning(
            "POST /extract model output was not valid JSON (first 200 chars): %r",
            raw[:200],
        )

    log.info(
        "POST /extract success parsed=%s input_tokens=%d output_tokens=%d",
        parsed is not None,
        response.usage.input_tokens,
        response.usage.output_tokens,
    )

    return ExtractResponse(
        raw_json=raw,
        parsed=parsed,
        input_tokens=response.usage.input_tokens,
        output_tokens=response.usage.output_tokens,
    )


# ---------------------------------------------------------------------------
# Entry point for direct execution (not required for uvicorn)
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import uvicorn

uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)

---

## Self-Check

Answer these without running code first:

**1. What is the root cause and the correct fix?**

- A. The Pydantic models are too large; reduce the number of response fields
- B. The client is being created per-request, causing connection overhead and memory accumulation; create the client once in the lifespan event and reuse it via app.state
- C. FastAPI's async handlers are not suited for synchronous Anthropic SDK calls; switch to a synchronous Flask app
- D. The /generate endpoint needs a connection pool size parameter passed to each client

**2. What went wrong in the service design?**

- A. The service correctly returns 500 for empty input; that is the expected behavior
- B. The empty check is happening inside the route handler with a manual raise, instead of being a Pydantic Field constraint (min_length=1) that produces 422 automatically
- C. FastAPI does not support 422 responses; all validation errors must be returned as 400
- D. The Anthropic SDK raises a 500 internally when it receives an empty string

**3. What is wrong with the health check design?**

- A. The health check should use a different model with lower latency
- B. The health check interval of 15 seconds is too frequent; increase it to 5 minutes
- C. The health check is testing the upstream API dependency, which causes the instance to appear unhealthy when the real service process is running fine; health checks should only verify that the service process itself is alive
- D. The health check should cache the model call result for 60 seconds to reduce load

**4. What change to the startup code would have prevented this deployment from accepting traffic?**

- A. Add a warning log at startup if ANTHROPIC_API_KEY is missing, but continue running
- B. Validate ANTHROPIC_API_KEY in the lifespan startup block and raise EnvironmentError if it is absent, which prevents the service from starting
- C. Check for the key inside each route handler and return 401 Unauthorized if it is missing
- D. Add a /config endpoint that shows which env vars are set

**5. What is the correct handling strategy?**

- A. Switch to a more expensive model that always returns raw JSON
- B. Add retry logic: if JSONDecodeError occurs, call the model again
- C. Strip markdown code fences before parsing, catch JSONDecodeError, and return the raw string with parsed=null instead of raising an exception
- D. Use a system prompt that says 'do not use markdown' and rely on it to fix the problem

**6. Why is showing users the generic message the correct behavior?**

- A. It is not correct; users should see the full error so they can report it accurately
- B. Exposing raw exception messages leaks internal details (file paths, library versions, API key fragments) that attackers can use; users receive a safe generic message while the full detail is in the internal logs
- C. FastAPI automatically hides error messages; there is no way to show more detail
- D. The generic message reduces customer support tickets because users give up and retry

**7. What is the most likely cause of this blocking behavior?**

- A. The Anthropic client is not thread-safe and serializes requests
- B. The route handlers are defined as synchronous `def` instead of `async def`, causing FastAPI to run them in a thread pool that becomes saturated with slow summarize requests
- C. The Pydantic response models are too large and serialization blocks the event loop
- D. app.state is not thread-safe and causes a lock contention issue

**8. What is the main production risk of returning raw text instead of a structured Pydantic response model?**

- A. Raw text responses cannot be returned from FastAPI endpoints
- B. Every downstream caller has to parse the text themselves, and any change to the model's phrasing silently breaks all integrations without any version mismatch or schema validation error
- C. Raw text uses more bandwidth than JSON for short strings
- D. FastAPI automatically converts all responses to JSON, so raw text is not actually possible

_Answers are in `checks.json` in the lesson directory._